In [ ]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans

MODEL_FILE = "model.pkl"
PIPELINE_FILE="pipeline.pkl"

data = pd.read_csv("Delhi_v2.csv")


data = data.drop(columns=["Unnamed: 0", "Price_sqft", "desc", "Landmarks", "Lift"], errors="ignore")

data["parking"] = data["parking"].fillna(0)
data["Balcony"] = data["Balcony"].fillna(0)

mode_value = data["Status"].mode()[0]
data["Status"] = data["Status"].fillna(mode_value)
data["Furnished_status"] = data["Furnished_status"].fillna("unknown")


geo = data[["latitude", "longitude"]]

scaler = StandardScaler()
geo_scaled = scaler.fit_transform(geo)

kmeans = KMeans(n_clusters=4, random_state=42)
data["location_cluster"] = kmeans.fit_predict(geo_scaled)

data = data.drop(columns=["latitude", "longitude"])


data["log_price"] = np.log(data["price"])

y = data["log_price"]
X = data.drop(columns=["log_price", "price"])   # price also remove


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



num_attributes = ["area", "Bedrooms", "Bathrooms", "Balcony", "parking"]

categorical_cols = [
    "Status",
    "Furnished_status",
    "type_of_building",
    "neworold",
    "location_cluster"
]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_attributes),
    ("cat", cat_pipeline, categorical_cols)
])


model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])


model_pipeline.fit(X_train, y_train)


import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
model_path = os.path.join(BASE_DIR, MODEL_FILE)

joblib.dump(model_pipeline, model_path)

print("Model trained and saved successfully.")








# import streamlit as st
# import joblib
# import pandas as pd
# import numpy as np

# model = joblib.load("model.pkl")

# st.title("House Price Prediction App")

# area = st.number_input("Area (sqft)")
# bedrooms = st.number_input("Bedrooms", step=1)
# bathrooms = st.number_input("Bathrooms", step=1)
# balcony = st.number_input("Balcony", step=1)
# parking = st.number_input("Parking (0/1)", step=1)

# status = st.selectbox("Status", ["Ready_to_move", "Under_Construction"])
# furnished = st.selectbox("Furnished Status", ["Furnished", "Semi-Furnished", "Unfurnished"])
# building = st.selectbox("Type of Building", ["Flat", "Builder_Floor", "Apartment"])
# neworold = st.selectbox("New or Old", ["New Property", "Old Property"])
# location_cluster = st.number_input("Location Cluster (0-3)", step=1)

# if st.button("Predict"):
#     input_data = pd.DataFrame({
#         "area": [area],
#         "Bedrooms": [bedrooms],
#         "Bathrooms": [bathrooms],
#         "Balcony": [balcony],
#         "parking": [parking],
#         "Status": [status],
#         "Furnished_status": [furnished],
#         "type_of_building": [building],
#         "neworold": [neworold],
#         "location_cluster": [location_cluster]
#     })

#     pred_log = model.predict(input_data)
#     pred_actual = np.exp(pred_log)

#     st.success(f"Predicted Price:  {round(pred_actual[0],2)}")